# Exploring the Hackathon Data

This notebook walks through every data source available for the hackathon, shows the schema, and demonstrates how to load, visualize, and merge the datasets.

**Prerequisites:** Run `python download_data.py` first to populate `data/train/`.

## Section 1: Setup & Data Loading

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import backtester modules if needed
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import glob
import sqlite3

import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

# Change this to "data/validation/" when testing on the validation set
DATA_DIR = Path("data/train/")

# Path to the SQLite database (contains market_prices, rtds_prices, market_outcomes)
DB_PATH = DATA_DIR / "polymarket.db"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR.resolve()}")
print(f"DB exists:    {DB_PATH.exists()}")

In [ ]:
# Connect to the database and list all tables
conn = sqlite3.connect(str(DB_PATH))
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tables in polymarket.db:")
print(tables.to_string(index=False))

## Section 2: Market Prices (Polymarket)

The `market_prices` table contains YES/NO prices, bids, asks, volume, and liquidity for every active BTC market, sampled roughly once per second.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in **microseconds** (divide by 1e6 for seconds) |
| `interval` | Market interval: `"5m"`, `"15m"`, or `"hourly"` |
| `market_slug` | Unique market identifier (e.g. `btc-updown-15m-1717243200`) |
| `yes_price` | Last traded YES token price (0-1) |
| `no_price` | Last traded NO token price (0-1) |
| `yes_bid` | Best bid for YES token |
| `yes_ask` | Best ask for YES token |
| `no_bid` | Best bid for NO token |
| `no_ask` | Best ask for NO token |
| `volume` | Cumulative market volume (USD) |
| `liquidity` | Current market liquidity (USD) |

In [ ]:
# Preview the market_prices table
prices_sample = pd.read_sql_query("SELECT * FROM market_prices LIMIT 5", conn)
prices_sample

In [ ]:
# Count rows per interval
interval_counts = pd.read_sql_query(
    "SELECT interval, COUNT(*) AS row_count FROM market_prices GROUP BY interval",
    conn,
)
interval_counts

In [ ]:
# Plot YES price over time for one 15-minute market
# Pick the first 15m market slug that has enough data
slug_row = pd.read_sql_query(
    """SELECT market_slug, COUNT(*) AS cnt
       FROM market_prices
       WHERE interval = '15m'
       GROUP BY market_slug
       ORDER BY cnt DESC
       LIMIT 1""",
    conn,
)
example_slug = slug_row["market_slug"].iloc[0]
print(f"Plotting market: {example_slug}")

market_df = pd.read_sql_query(
    "SELECT timestamp_us, yes_price FROM market_prices WHERE market_slug = ? ORDER BY timestamp_us",
    conn,
    params=[example_slug],
)

# Convert microseconds to datetime
market_df["time"] = pd.to_datetime(market_df["timestamp_us"] / 1e6, unit="s", utc=True)

plt.figure(figsize=(12, 4))
plt.plot(market_df["time"], market_df["yes_price"], linewidth=0.8)
plt.title(f"YES Price — {example_slug}")
plt.xlabel("Time (UTC)")
plt.ylabel("YES Price")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 3: Chainlink Oracle Prices (rtds_prices)

The `rtds_prices` table stores on-chain Chainlink BTC/USD oracle prices. **This is the reference price used for market settlement** -- if Chainlink BTC at close >= Chainlink BTC at open, YES wins.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in **microseconds** |
| `source` | Price source (filter for `"chainlink"`) |
| `symbol` | Asset symbol (e.g. `"BTC"`) |
| `price` | BTC/USD price from Chainlink oracle |

In [ ]:
# Preview Chainlink prices
chainlink_sample = pd.read_sql_query(
    "SELECT * FROM rtds_prices WHERE source='chainlink' LIMIT 5", conn
)
chainlink_sample

In [ ]:
# Plot Chainlink BTC price over time
chainlink_df = pd.read_sql_query(
    "SELECT timestamp_us, price FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us",
    conn,
)
chainlink_df["time"] = pd.to_datetime(chainlink_df["timestamp_us"] / 1e6, unit="s", utc=True)

plt.figure(figsize=(12, 4))
plt.plot(chainlink_df["time"], chainlink_df["price"], linewidth=0.5, color="orange")
plt.title("Chainlink BTC/USD Oracle Price")
plt.xlabel("Time (UTC)")
plt.ylabel("Price (USD)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 4: Order Books (polymarket_books/)

Order book snapshots are stored as CSV files in `polymarket_books/`, one file per day. Each row is a snapshot of the full order book for one market at one point in time.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in microseconds |
| `interval` | Market interval (`"5m"`, `"15m"`, `"hourly"`) |
| `market_slug` | Market identifier |
| `yes_best_bid` | Best bid price for YES token |
| `yes_best_ask` | Best ask price for YES token |
| `no_best_bid` | Best bid price for NO token |
| `no_best_ask` | Best ask price for NO token |
| `yes_bids_json` | Full YES bid depth as JSON: `[[price, size], ...]` |
| `yes_asks_json` | Full YES ask depth as JSON: `[[price, size], ...]` |
| `no_bids_json` | Full NO bid depth as JSON |
| `no_asks_json` | Full NO ask depth as JSON |
| `yes_n_bids`, `yes_n_asks` | Number of bid/ask levels for YES |
| `no_n_bids`, `no_n_asks` | Number of bid/ask levels for NO |
| `yes_total_bid_size`, `yes_total_ask_size` | Total depth on each side for YES |
| `no_total_bid_size`, `no_total_ask_size` | Total depth on each side for NO |

In [ ]:
# Load order book CSV files (filenames vary by date)
books_dir = DATA_DIR / "polymarket_books"
book_files = sorted(books_dir.glob("*.csv"))
print(f"Found {len(book_files)} order book files:")
for f in book_files[:5]:
    print(f"  {f.name}")

# Load the first file as an example
if book_files:
    books_df = pd.read_csv(book_files[0])
    print(f"\nShape: {books_df.shape}")
    print(f"Columns: {list(books_df.columns)}")
    books_df.head(3)

In [ ]:
# Parse the full order book depth from JSON columns
# Each JSON column contains a list of [price, size] pairs
if book_files and not books_df.empty:
    example_row = books_df.iloc[0]
    print(f"Market: {example_row['market_slug']}")
    print(f"\nYES bids (top of book first):")
    yes_bids = json.loads(example_row["yes_bids_json"])
    for price, size in yes_bids[:5]:
        print(f"  ${price:.2f}  x  {size:.1f} shares")

    print(f"\nYES asks (cheapest first):")
    yes_asks = json.loads(example_row["yes_asks_json"])
    for price, size in yes_asks[:5]:
        print(f"  ${price:.2f}  x  {size:.1f} shares")

In [ ]:
# Plot bid-ask spread over time for one market
if book_files:
    # Load all book files and concatenate
    all_books = pd.concat([pd.read_csv(f) for f in book_files], ignore_index=True)

    # Pick a 15m market with the most snapshots
    book_counts = all_books[all_books["interval"] == "15m"].groupby("market_slug").size()
    if not book_counts.empty:
        top_slug = book_counts.idxmax()
        slug_books = all_books[all_books["market_slug"] == top_slug].copy()
        slug_books["time"] = pd.to_datetime(slug_books["timestamp_us"] / 1e6, unit="s", utc=True)
        slug_books["yes_spread"] = slug_books["yes_best_ask"] - slug_books["yes_best_bid"]

        plt.figure(figsize=(12, 4))
        plt.plot(slug_books["time"], slug_books["yes_spread"], linewidth=0.8, color="green")
        plt.title(f"YES Bid-Ask Spread — {top_slug}")
        plt.xlabel("Time (UTC)")
        plt.ylabel("Spread")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Section 5: Binance LOB (binance_lob/)

Binance BTCUSDT order book depth snapshots stored as hourly Parquet files. These provide 20-level depth at approximately 100ms resolution -- much higher frequency than the Polymarket data.

| Column | Description |
|---|---|
| `timestamp_us` | Unix timestamp in microseconds |
| `event_time_ms` | Binance event timestamp in milliseconds |
| `ask_price_1` .. `ask_price_20` | Ask prices at levels 1-20 |
| `ask_vol_1` .. `ask_vol_20` | Ask volumes at levels 1-20 |
| `bid_price_1` .. `bid_price_20` | Bid prices at levels 1-20 |
| `bid_vol_1` .. `bid_vol_20` | Bid volumes at levels 1-20 |
| `mid_price` | (best bid + best ask) / 2 |
| `spread` | best ask - best bid |

**Note:** 20-level depth at ~100ms resolution means ~36,000 rows per hour.

In [ ]:
# Load Binance LOB Parquet files (filenames are YYYY-MM-DD_HH.parquet)
binance_dir = DATA_DIR / "binance_lob"
parquet_files = sorted(binance_dir.glob("*.parquet"))
print(f"Found {len(parquet_files)} Binance LOB files:")
for f in parquet_files[:5]:
    print(f"  {f.name}")

# Load one file to explore
if parquet_files:
    binance_sample = pd.read_parquet(parquet_files[0])
    print(f"\nShape: {binance_sample.shape}")
    print(f"Columns: {list(binance_sample.columns)}")
    binance_sample.head(3)

In [ ]:
# Plot mid_price and spread from one hour of Binance data
if parquet_files:
    binance_sample["time"] = pd.to_datetime(
        binance_sample["timestamp_us"] / 1e6, unit="s", utc=True
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(binance_sample["time"], binance_sample["mid_price"], linewidth=0.5)
    ax1.set_title("Binance BTCUSDT Mid Price (1 hour)")
    ax1.set_ylabel("Price (USD)")
    ax1.grid(True, alpha=0.3)

    ax2.plot(binance_sample["time"], binance_sample["spread"], linewidth=0.5, color="red")
    ax2.set_title("Binance BTCUSDT Spread")
    ax2.set_xlabel("Time (UTC)")
    ax2.set_ylabel("Spread (USD)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Section 6: Merging Datasets

All three data sources use `timestamp_us` (microseconds). To align them, convert to integer seconds and merge. The backtester does this automatically via `build_timeline()`, but this is useful for exploratory analysis.

**Important:** Forward-fill (`ffill`) is used because data sources update at different frequencies -- Binance updates every ~100ms, Chainlink every ~1s, and Polymarket prices every ~1s but only while a market is active.

In [ ]:
# Load all three sources and align to 1-second resolution

# 1. Market prices from SQLite
prices_df = pd.read_sql_query(
    "SELECT * FROM market_prices ORDER BY timestamp_us", conn
)
prices_df["ts_sec"] = prices_df["timestamp_us"] // 1_000_000

# 2. Chainlink prices from SQLite
chainlink_df = pd.read_sql_query(
    "SELECT * FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us", conn
)
chainlink_df["ts_sec"] = chainlink_df["timestamp_us"] // 1_000_000

# 3. Binance mid prices (aggregate to 1-second: take last value per second)
if parquet_files:
    # Load a subset of files to keep memory manageable
    binance_frames = [pd.read_parquet(f) for f in parquet_files[:3]]
    binance_df = pd.concat(binance_frames, ignore_index=True)
    binance_df["ts_sec"] = binance_df["timestamp_us"] // 1_000_000
    # Downsample to 1-second: take the last snapshot per second
    binance_1s = binance_df.groupby("ts_sec").agg(
        mid_price=("mid_price", "last"),
        spread=("spread", "last"),
    ).reset_index()
else:
    binance_1s = pd.DataFrame(columns=["ts_sec", "mid_price", "spread"])

# Merge on ts_sec
merged = prices_df.merge(
    chainlink_df[["ts_sec", "price"]].rename(columns={"price": "chainlink_btc"}),
    on="ts_sec",
    how="left",
)
merged = merged.merge(
    binance_1s[["ts_sec", "mid_price"]].rename(columns={"mid_price": "binance_btc"}),
    on="ts_sec",
    how="left",
)

# Forward-fill external prices (they update at different rates)
merged[["chainlink_btc", "binance_btc"]] = merged[["chainlink_btc", "binance_btc"]].ffill()

print(f"Merged shape: {merged.shape}")
merged[["timestamp_us", "market_slug", "yes_price", "chainlink_btc", "binance_btc"]].head(10)

## Section 7: Understanding Market Settlement

Each market resolves based on the **Chainlink oracle price**:
- Extract the market's start timestamp from its slug (e.g. `btc-updown-15m-1717243200` starts at unix time `1717243200`)
- The market's end time is `start + interval_seconds` (300 for 5m, 900 for 15m, 3600 for hourly)
- **YES wins** if Chainlink BTC price at end >= Chainlink BTC price at start
- **NO wins** if Chainlink BTC price at end < Chainlink BTC price at start

The `market_outcomes` table (if available) stores the resolved outcomes directly.

In [ ]:
# Check if market_outcomes table exists and preview it
has_outcomes = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='market_outcomes'", conn
)

if not has_outcomes.empty:
    outcomes_df = pd.read_sql_query("SELECT * FROM market_outcomes LIMIT 5", conn)
    print("market_outcomes table found:")
    display(outcomes_df)
else:
    print("market_outcomes table not found in this dataset.")
    print("You can compute outcomes from Chainlink prices instead (see below).")

In [ ]:
# Compute settlement outcome from Chainlink prices for a 15m market
import re

# Get all unique 15m slugs
slugs_15m = prices_df[prices_df["interval"] == "15m"]["market_slug"].unique()

# Load full Chainlink series for lookups
cl_full = pd.read_sql_query(
    "SELECT timestamp_us, price FROM rtds_prices WHERE source='chainlink' ORDER BY timestamp_us",
    conn,
)
cl_full["ts_sec"] = cl_full["timestamp_us"] // 1_000_000

results = []
for slug in slugs_15m[:10]:  # Check first 10 markets
    m = re.match(r"btc-updown-15m-(\d+)", slug)
    if not m:
        continue
    start_ts = int(m.group(1))
    end_ts = start_ts + 900  # 15 minutes

    # Find Chainlink price nearest to start and end
    start_diff = (cl_full["ts_sec"] - start_ts).abs()
    end_diff = (cl_full["ts_sec"] - end_ts).abs()

    if start_diff.min() > 30 or end_diff.min() > 30:
        continue  # Skip if no Chainlink data within 30 seconds

    open_price = cl_full.loc[start_diff.idxmin(), "price"]
    close_price = cl_full.loc[end_diff.idxmin(), "price"]
    outcome = "YES" if close_price >= open_price else "NO"

    results.append({
        "slug": slug,
        "open_price": open_price,
        "close_price": close_price,
        "change": close_price - open_price,
        "outcome": outcome,
    })

if results:
    pd.DataFrame(results)

## Section 8: Key Notes for Strategy Development

- **The backtester merges all data automatically** -- you do not need to do the merging above yourself. Just implement `on_tick()` and the engine feeds you a `MarketState` every second.
- `state.btc_mid` = Binance BTCUSDT mid price (high-frequency reference)
- `state.chainlink_btc` = Chainlink on-chain oracle price (**used for settlement**)
- **Settlement uses Chainlink prices, not Binance.** The two can diverge by several dollars.
- Your strategy chooses which intervals to trade by filtering `state.markets` by `market.interval` (e.g. `"5m"`, `"15m"`, `"hourly"`).
- The backtester loads ALL intervals by default -- your strategy decides what to trade.
- **Order execution walks the real order book** -- if you buy 500 shares, that liquidity is consumed from the book. Large orders will get worse average prices.
- Orders submitted at tick T execute at tick T+1 (1-second latency).
- **Position limit:** 500 shares per token per market.
- **No short selling** -- you can only sell tokens you own.
- YES price + NO price should be approximately $1. Any deviation is a potential arbitrage signal.

In [ ]:
# Clean up
conn.close()